# 03 — HAR Model Training (1D-CNN)

**Tareas A2, A3, A4, A5 — Íñigo**

Pipeline:
1. Cargar datos procesados de `01_eda_activities.ipynb`
2. Baseline: Random Forest + SVM con features manuales (A2)
3. 1D-CNN — arquitectura del plan (A3)
4. Ablación CNN+GRU opcional (A4)
5. Evaluación LOSO — matriz de confusión, F1 por clase (A5)

In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

sys.path.insert(0, str(Path('..').resolve()))
from src.preprocessing.windowing import augment_window

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score, accuracy_score
)
from sklearn.model_selection import LeaveOneGroupOut, train_test_split

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

DATA_PROC  = Path('../data/processed')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(exist_ok=True)

print(f'TensorFlow {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

TensorFlow 2.21.0
GPU available: True


In [ ]:
import mlflow
import mlflow.sklearn
import mlflow.keras

# ── MLflow setup ──────────────────────────────────────────────────────────────
MLFLOW_URI = "http://localhost:5000"
EXPERIMENT_NAME = "vitalia-har-cnn"

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"MLflow tracking URI: {MLFLOW_URI}")
print(f"Experiment: {EXPERIMENT_NAME}")

In [15]:
# Load preprocessed data (output of notebook 01)
X = np.load(DATA_PROC / 'X_activities.npy')        # (N, 128, 6)
y = np.load(DATA_PROC / 'y_activities.npy')         # (N,) labels 1-6
subjects = np.load(DATA_PROC / 'subjects_activities.npy')  # (N,)

# Convert labels to 0-indexed
y_0 = y - 1  # 0=walking, 1=upstairs, 2=downstairs, 3=sitting, 4=standing, 5=running
N_CLASSES = len(np.unique(y_0))
CLASS_NAMES = ['walking', 'upstairs', 'downstairs', 'sitting', 'standing', 'running']

print(f'X={X.shape}, y={y_0.shape}, subjects={len(np.unique(subjects))}')
print(f'Classes: {N_CLASSES}')

X=(29894, 128, 6), y=(29894,), subjects=54
Classes: 6


## A2 — Baseline: Random Forest + SVM

Features manuales: media, std, energía, zero-crossing rate, correlación inter-eje (accel)

In [16]:
def extract_features(X_windows: np.ndarray) -> np.ndarray:
    """Extract handcrafted features from (N, 128, 6) windows."""
    N = X_windows.shape[0]
    feats = []
    for i in range(N):
        w = X_windows[i]  # (128, 6)
        f = []
        # Per-channel: mean, std, energy, zero-crossing rate
        for ch in range(6):
            s = w[:, ch]
            f.extend([
                s.mean(),
                s.std(),
                (s ** 2).mean(),                         # energy
                ((s[:-1] * s[1:]) < 0).sum() / 128.0,   # zero-crossing rate
            ])
        # Accel inter-axis correlations (3 pairs)
        accel = w[:, :3]
        for a, b in [(0,1), (0,2), (1,2)]:
            f.append(np.corrcoef(accel[:, a], accel[:, b])[0, 1])
        # SVM mean and max
        svm = np.sqrt((accel ** 2).sum(axis=1))
        f.extend([svm.mean(), svm.max()])
        feats.append(f)
    return np.array(feats, dtype=np.float32)

print('Extracting features...')
X_feat = extract_features(X)
print(f'Feature matrix: {X_feat.shape}')

Extracting features...
Feature matrix: (29894, 29)


In [17]:
# LOSO evaluation for RF and SVM baselines
logo = LeaveOneGroupOut()
rf_scores, svm_scores = [], []

unique_subjects = np.unique(subjects)
# Cap at 10 subjects for speed; use all for final eval
eval_subjects = unique_subjects[:10]
mask = np.isin(subjects, eval_subjects)
X_f_sub, y_sub, subj_sub = X_feat[mask], y_0[mask], subjects[mask]

scaler = StandardScaler()

for train_idx, test_idx in logo.split(X_f_sub, y_sub, groups=subj_sub):
    X_tr = scaler.fit_transform(X_f_sub[train_idx])
    X_te = scaler.transform(X_f_sub[test_idx])
    y_tr, y_te = y_sub[train_idx], y_sub[test_idx]

    rf = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
    rf.fit(X_tr, y_tr)
    rf_scores.append(f1_score(y_te, rf.predict(X_te), average='macro'))

    svm = SVC(kernel='rbf', C=1.0, gamma='scale')
    svm.fit(X_tr, y_tr)
    svm_scores.append(f1_score(y_te, svm.predict(X_te), average='macro'))

print(f'RF  LOSO F1-macro (mean ± std): {np.mean(rf_scores):.3f} ± {np.std(rf_scores):.3f}')
print(f'SVM LOSO F1-macro (mean ± std): {np.mean(svm_scores):.3f} ± {np.std(svm_scores):.3f}')

RF  LOSO F1-macro (mean ± std): 0.826 ± 0.068
SVM LOSO F1-macro (mean ± std): 0.810 ± 0.087


In [ ]:
# ── MLflow logging — baseline models (RF + SVM) ───────────────────────────────
# Buena práctica: cada modelo en su propio run bajo el mismo experimento.
# Esto permite comparar todos los modelos (baseline + CNN) en una sola vista.

for name, scores, all_preds_bl in [
    ("random_forest_baseline", rf_scores, None),
    ("svm_baseline", svm_scores, None),
]:
    with mlflow.start_run(run_name=name):
        mlflow.log_params({
            "model_type":     name.split("_")[0].upper() + "_" + name.split("_")[1].upper(),
            "cv_strategy":    "LOSO",
            "n_subjects_eval": len(eval_subjects),
            "n_features":     X_feat.shape[1],
            "dataset":        "UCI_HAR_240+MotionSense+PAMAP2",
            "class_weight":   "balanced",
        })
        mlflow.log_metrics({
            "f1_macro_mean": float(np.mean(rf_scores if "forest" in name else svm_scores)),
            "f1_macro_std":  float(np.std(rf_scores if "forest" in name else svm_scores)),
        })
    print(f"MLflow run logged: {name}")

## A3 — 1D-CNN

## A3-v2 — ResNet1D (mejoras sobre v1)

Cambios respecto al modelo base:
- **Arquitectura residual**: bloques con shortcut connections → gradientes más estables en profundidad
- **Stem kernel=7**: captura ~140 ms de contexto temporal en la primera capa
- **3 bloques residuales** (64 → 128 → 256 filtros) con projection shortcuts
- **Augmentación por clase**: running ×5, upstairs/downstairs ×2 → clases balanceadas
- **Class weights**: penalización extra en errores de running (peso 3.5×)
- **Cosine LR + warmup**: LR más suave, sin saltos bruscos de ReduceLROnPlateau
- Input/output sin cambios: `(128, 6)` → `(6,)` — compatible con Flutter sin modificaciones


In [ ]:
# HAR Model v2 — Residual 1D-CNN
# Input: (batch, 128, 6) — accel_xyz + gyro_xyz @ 50 Hz
# Output: (batch, 6) — class probabilities. Same shape as v1 → Flutter-compatible.
from tensorflow.keras import layers as L

def build_har_model_v2(
    n_classes: int, window_size: int = 128, n_channels: int = 6
) -> keras.Model:
    inputs = keras.Input(shape=(window_size, n_channels), name='sensor_input')

    # Stem: k=7 captures ~140 ms temporal context
    x = L.Conv1D(64, kernel_size=7, padding='same', use_bias=False, name='stem_conv')(inputs)
    x = L.BatchNormalization(name='stem_bn')(x)
    x = L.Activation('relu', name='stem_act')(x)

    # Residual block 1 — 64 filters
    shortcut = x
    x = L.Conv1D(64, 3, padding='same', use_bias=False, name='res1_c1')(x)
    x = L.BatchNormalization(name='res1_bn1')(x)
    x = L.Activation('relu', name='res1_a1')(x)
    x = L.Conv1D(64, 3, padding='same', use_bias=False, name='res1_c2')(x)
    x = L.BatchNormalization(name='res1_bn2')(x)
    x = L.Add(name='res1_add')([x, shortcut])
    x = L.Activation('relu', name='res1_out')(x)
    x = L.MaxPooling1D(2, name='pool1')(x)  # 128 → 64

    # Residual block 2 — 128 filters (projection shortcut)
    shortcut = L.Conv1D(128, 1, padding='same', use_bias=False, name='res2_proj')(x)
    x = L.Conv1D(128, 3, padding='same', use_bias=False, name='res2_c1')(x)
    x = L.BatchNormalization(name='res2_bn1')(x)
    x = L.Activation('relu', name='res2_a1')(x)
    x = L.Conv1D(128, 3, padding='same', use_bias=False, name='res2_c2')(x)
    x = L.BatchNormalization(name='res2_bn2')(x)
    x = L.Add(name='res2_add')([x, shortcut])
    x = L.Activation('relu', name='res2_out')(x)
    x = L.MaxPooling1D(2, name='pool2')(x)  # 64 → 32

    # Residual block 3 — 256 filters (projection shortcut)
    shortcut = L.Conv1D(256, 1, padding='same', use_bias=False, name='res3_proj')(x)
    x = L.Conv1D(256, 3, padding='same', use_bias=False, name='res3_c1')(x)
    x = L.BatchNormalization(name='res3_bn1')(x)
    x = L.Activation('relu', name='res3_a1')(x)
    x = L.Conv1D(256, 3, padding='same', use_bias=False, name='res3_c2')(x)
    x = L.BatchNormalization(name='res3_bn2')(x)
    x = L.Add(name='res3_add')([x, shortcut])
    x = L.Activation('relu', name='res3_out')(x)

    # Classifier head
    x = L.GlobalAveragePooling1D(name='gap')(x)
    x = L.Dense(128, activation='relu', name='fc1')(x)
    x = L.Dropout(0.4, name='drop')(x)
    outputs = L.Dense(n_classes, activation='softmax', name='class_probs')(x)

    return keras.Model(inputs, outputs, name='HAR_ResNet1D_v2')


# Backward-compat alias (cells that already call build_1d_cnn still work)
build_1d_cnn = build_har_model_v2

model = build_har_model_v2(N_CLASSES)
model.summary()


In [ ]:
# Full LOSO CNN v2 training — per-class augmentation + class weights + cosine LR
QUICK_RUN = False

import os, math
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Per-class augmentation multipliers (0-indexed: 0=walk,1=up,2=down,3=sit,4=stand,5=run)
AUG_MULTIPLIERS = {0: 1, 1: 2, 2: 2, 3: 1, 4: 1, 5: 5}

# Class weights — higher penalty for running errors
CLASS_WEIGHT = {0: 1.0, 1: 1.8, 2: 2.1, 3: 1.0, 4: 1.1, 5: 3.5}


def augment_per_class(X_tr, y_tr):
    """Augment minority classes by AUG_MULTIPLIERS[cls] - 1 extra copies."""
    X_parts, y_parts = [X_tr], [y_tr]
    for cls, mult in AUG_MULTIPLIERS.items():
        if mult <= 1:
            continue
        idx = np.where(y_tr == cls)[0]
        if len(idx) == 0:
            continue
        for _ in range(mult - 1):
            wins = [augment_window(X_tr[i], n_augments=1)[0] for i in idx]
            X_parts.append(np.stack(wins))
            y_parts.append(np.full(len(idx), cls, dtype=y_tr.dtype))
    return np.concatenate(X_parts), np.concatenate(y_parts)


def cosine_lr(epoch, total=80, warmup=5, lr_max=1e-3, lr_min=1e-5):
    if epoch < warmup:
        return lr_max * (epoch + 1) / warmup
    t = (epoch - warmup) / (total - warmup)
    return float(lr_min + 0.5 * (lr_max - lr_min) * (1 + math.cos(math.pi * t)))


unique_subjects = np.unique(subjects)
if QUICK_RUN:
    unique_subjects = unique_subjects[:3]

cnn_fold_results = []
all_y_true, all_y_pred = [], []

for fold_idx, test_subj in enumerate(unique_subjects):
    print(f'Fold {fold_idx+1}/{len(unique_subjects)} — test subject {test_subj}')

    test_mask  = subjects == test_subj
    train_mask = ~test_mask

    X_tr, y_tr = X[train_mask], y_0[train_mask]
    X_te, y_te = X[test_mask],  y_0[test_mask]

    # Per-class augmentation
    X_tr, y_tr = augment_per_class(X_tr, y_tr)

    # Stratified validation split
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_tr, y_tr, test_size=0.1, stratify=y_tr, random_state=42
    )

    fold_model = build_har_model_v2(N_CLASSES)
    fold_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )

    callbacks = [
        keras.callbacks.LearningRateScheduler(cosine_lr, verbose=0),
        keras.callbacks.EarlyStopping(
            patience=15, restore_best_weights=True, verbose=0
        ),
    ]

    fold_model.fit(
        X_tr, y_tr,
        epochs=80,
        batch_size=64,
        validation_data=(X_val, y_val),
        class_weight=CLASS_WEIGHT,
        callbacks=callbacks,
        verbose=0,
    )

    y_pred = fold_model.predict(X_te, verbose=0).argmax(axis=1)
    all_y_true.extend(y_te)
    all_y_pred.extend(y_pred)
    f1  = f1_score(y_te, y_pred, average='macro')
    acc = accuracy_score(y_te, y_pred)
    cnn_fold_results.append({'subject': test_subj, 'f1_macro': f1, 'accuracy': acc})
    print(f'  F1-macro={f1:.3f}  acc={acc:.3f}')

all_y_true = np.array(all_y_true)
all_y_pred = np.array(all_y_pred)
LOSO_N_SUBJECTS = len(unique_subjects)

df_loso = pd.DataFrame(cnn_fold_results)
print('\n=== LOSO CNN v2 Results ===')
print(df_loso.describe())

np.save(DATA_PROC / 'loso_y_true.npy', all_y_true)
np.save(DATA_PROC / 'loso_y_pred.npy', all_y_pred)
df_loso.to_csv(DATA_PROC / 'loso_fold_results.csv', index=False)
print('LOSO predictions saved to data/processed/')


In [ ]:
# Train final model v2 on ALL data for TFLite export
print('Training final HAR model v2 on full dataset...')
final_model = build_har_model_v2(N_CLASSES)
final_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

# Per-class augmentation on full dataset
X_full, y_full = augment_per_class(X, y_0)

# Log class distribution after augmentation
unique_cls, cls_counts = np.unique(y_full, return_counts=True)
print('Post-augmentation class distribution:')
for cls, cnt in zip(unique_cls, cls_counts):
    print(f'  {CLASS_NAMES[cls]:12s}: {cnt:6d} windows')

# Stratified validation split
X_full, X_val, y_full, y_val = train_test_split(
    X_full, y_full, test_size=0.05, stratify=y_full, random_state=42
)
print(f'Train: {len(X_full)}  Val: {len(X_val)}')

final_model.fit(
    X_full, y_full,
    epochs=80,
    batch_size=64,
    validation_data=(X_val, y_val),
    class_weight=CLASS_WEIGHT,
    callbacks=[
        keras.callbacks.LearningRateScheduler(cosine_lr, verbose=0),
        keras.callbacks.EarlyStopping(
            patience=15, restore_best_weights=True, verbose=1
        ),
        keras.callbacks.ModelCheckpoint(
            str(MODELS_DIR / 'har_model_keras.keras'),
            save_best_only=True, monitor='val_accuracy', verbose=1
        ),
    ],
    verbose=1,
)
print('Final v2 model saved to models/har_model_keras.keras')


## A4 — Ablation: CNN + GRU (optional)

In [ ]:
def build_cnn_gru(n_classes: int, window_size: int = 128, n_channels: int = 6) -> keras.Model:
    inp = keras.Input(shape=(window_size, n_channels))

    x = layers.Conv1D(64, kernel_size=3, activation='relu', padding='same')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)

    x = layers.Conv1D(128, kernel_size=3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)

    x = layers.Conv1D(128, kernel_size=3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)

    x = layers.GRU(64)(x)  # adds temporal context
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(n_classes, activation='softmax')(x)

    return keras.Model(inp, out, name='HAR_CNN_GRU')

gru_model = build_cnn_gru(N_CLASSES)
print(f'CNN      params: {final_model.count_params():,}')
print(f'CNN+GRU  params: {gru_model.count_params():,}')

In [ ]:
# A4 — Train CNN+GRU under the SAME LOSO protocol as the CNN, to compare F1 fairly.
# GRU is slower; cap folds with ABLATION_N_SUBJECTS for a quick read, or set to None for full LOSO.
ABLATION_N_SUBJECTS = None  # e.g. 10 for a fast ablation; None = all subjects

abl_subjects = np.unique(subjects)
if ABLATION_N_SUBJECTS is not None:
    abl_subjects = abl_subjects[:ABLATION_N_SUBJECTS]

gru_fold_results = []
gru_y_true, gru_y_pred = [], []

for fold_idx, test_subj in enumerate(abl_subjects):
    print(f'[GRU] Fold {fold_idx+1}/{len(abl_subjects)} — test subject {test_subj}')

    test_mask  = subjects == test_subj
    train_mask = ~test_mask
    X_tr, y_tr = X[train_mask], y_0[train_mask]
    X_te, y_te = X[test_mask],  y_0[test_mask]

    # Same augmentation as the CNN loop
    aug_idx = np.random.choice(len(X_tr), size=int(len(X_tr) * AUGMENT_FRAC), replace=False)
    aug_wins = [augment_window(X_tr[i], n_augments=1)[0] for i in aug_idx]
    X_tr = np.concatenate([X_tr, np.stack(aug_wins)])
    y_tr = np.concatenate([y_tr, y_tr[aug_idx]])

    # Stratified validation split (data grouped by class on disk)
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_tr, y_tr, test_size=0.1, stratify=y_tr, random_state=42
    )

    m = build_cnn_gru(N_CLASSES)
    m.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    m.fit(
        X_tr, y_tr, epochs=60, batch_size=64, validation_data=(X_val, y_val),
        callbacks=[
            keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, verbose=0),
            keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5, verbose=0),
        ],
        verbose=0,
    )
    y_pred = m.predict(X_te, verbose=0).argmax(axis=1)
    gru_y_true.extend(y_te)
    gru_y_pred.extend(y_pred)
    f1 = f1_score(y_te, y_pred, average='macro')
    acc = accuracy_score(y_te, y_pred)
    gru_fold_results.append({'subject': test_subj, 'f1_macro': f1, 'accuracy': acc})
    print(f'  F1-macro={f1:.3f}  acc={acc:.3f}')

gru_y_true = np.array(gru_y_true)
gru_y_pred = np.array(gru_y_pred)
GRU_N_SUBJECTS = len(abl_subjects)

df_gru = pd.DataFrame(gru_fold_results)
print('\n=== LOSO CNN+GRU Results ===')
print(df_gru.describe())
print(f'\nCNN+GRU LOSO F1-macro: {f1_score(gru_y_true, gru_y_pred, average="macro"):.3f} '
      f'({GRU_N_SUBJECTS} subjects)')

## A5 — Evaluation: confusion matrix + per-class F1

In [ ]:
# First eval cell — reload LOSO predictions from disk if the kernel was restarted.
# all_y_true / all_y_pred are produced and persisted by the LOSO loop (cell A3).
if "all_y_pred" not in globals():
    all_y_true = np.load(DATA_PROC / "loso_y_true.npy")
    all_y_pred = np.load(DATA_PROC / "loso_y_pred.npy")
    print(f"Loaded LOSO predictions from disk: {len(all_y_true)} windows")

if "LOSO_N_SUBJECTS" in globals():
    n_total = len(np.unique(subjects))
    if LOSO_N_SUBJECTS < n_total:
        print(
            f"WARNING: QUICK_RUN was on — matrix covers only "
            f"{LOSO_N_SUBJECTS}/{n_total} subjects. Re-run cell A3 with QUICK_RUN=False."
        )
    else:
        print(
            f"Full LOSO coverage: {LOSO_N_SUBJECTS}/{n_total} subjects, "
            f"{len(all_y_true)} windows."
        )

In [ ]:
# Per-class F1 + comparison table
report = classification_report(
    all_y_true, all_y_pred, target_names=CLASS_NAMES, output_dict=True
)
df_report = pd.DataFrame(report).T.iloc[:-3]  # drop avg rows for now

print(classification_report(all_y_true, all_y_pred, target_names=CLASS_NAMES))

# Comparison table: RF vs SVM vs 1D-CNN (+ CNN+GRU if the A4 ablation ran)
print("\n=== Model comparison (LOSO F1-macro) ===")
rows = [
    ("Random Forest", np.mean(rf_scores)),
    ("SVM (RBF)", np.mean(svm_scores)),
    ("1D-CNN", f1_score(all_y_true, all_y_pred, average="macro")),
]
if "gru_y_pred" in globals() and len(gru_y_pred) > 0:
    rows.append(("CNN+GRU", f1_score(gru_y_true, gru_y_pred, average="macro")))

comparison = pd.DataFrame(rows, columns=["Model", "F1-macro (mean)"])
print(comparison.to_string(index=False))

# Coverage note: RF/SVM use 10 subjects, CNN uses all; check GRU coverage matches before claiming.
if "GRU_N_SUBJECTS" in globals():
    print(
        f"\n(CNN: {LOSO_N_SUBJECTS} subj · CNN+GRU: {GRU_N_SUBJECTS} subj · RF/SVM: {len(eval_subjects)} subj)"
    )

In [ ]:
# MLflow logging — 1D-CNN v2 LOSO + model registry
from sklearn.metrics import f1_score as _f1, accuracy_score as _acc

df_loso_loaded = pd.read_csv(DATA_PROC / 'loso_fold_results.csv')

with mlflow.start_run(run_name='cnn1d_v2_loso') as run:
    mlflow.log_params({
        'architecture':       'ResNet1D-v2 (64-128-256, residual)',
        'window_size':        128,
        'overlap':            0.5,
        'channels':           'ax,ay,az,gx,gy,gz',
        'n_classes':          N_CLASSES,
        'aug_multipliers':    str(AUG_MULTIPLIERS),
        'class_weight':       str(CLASS_WEIGHT),
        'dataset':            'UCI_HAR_240+MotionSense+PAMAP2',
        'n_subjects_loso':    LOSO_N_SUBJECTS,
        'optimizer':          'Adam',
        'learning_rate':      '1e-3 cosine_warmup5',
        'batch_size':         64,
        'max_epochs':         80,
        'early_stop_patience': 15,
    })

    for i, row in df_loso_loaded.iterrows():
        mlflow.log_metric('fold_f1_macro', float(row['f1_macro']), step=i)
        mlflow.log_metric('fold_accuracy', float(row['accuracy']), step=i)

    f1_loso  = _f1(all_y_true, all_y_pred, average='macro')
    acc_loso = _acc(all_y_true, all_y_pred)
    mlflow.log_metrics({
        'f1_macro_mean':  float(df_loso_loaded.f1_macro.mean()),
        'f1_macro_std':   float(df_loso_loaded.f1_macro.std()),
        'accuracy_mean':  float(df_loso_loaded.accuracy.mean()),
        'f1_macro_full':  float(f1_loso),
        'accuracy_full':  float(acc_loso),
    })

    f1_per_class_arr = _f1(all_y_true, all_y_pred, average=None)
    for cls_name, f1_val in zip(CLASS_NAMES, f1_per_class_arr):
        mlflow.log_metric(f'f1_{cls_name}', float(f1_val))

    mlflow.log_artifact(str(DATA_PROC / 'har_confusion_matrix.png'))
    mlflow.log_artifact(str(DATA_PROC / 'har_f1_per_class.png'))

    mlflow.keras.log_model(final_model, 'har_model_v2')

    model_uri = f'runs:/{run.info.run_id}/har_model_v2'
    mlflow.register_model(model_uri, 'vitalia-har')

    print(f'MLflow run logged: cnn1d_v2_loso (run_id={run.info.run_id})')
    print(f'Model registered as vitalia-har in MLflow Model Registry')


In [ ]:
# Confusion matrix (aggregated over all LOSO folds)
cm = confusion_matrix(all_y_true, all_y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('HAR 1D-CNN — Normalised confusion matrix (LOSO)')
ax.set_title('HAR 1D-CNN — Normalised confusion matrix (LOSO)')
plt.tight_layout()
plt.savefig(DATA_PROC / 'har_confusion_matrix.png', dpi=120)
plt.show()

In [ ]:
# Per-class F1 bar chart
f1_per_class = f1_score(all_y_true, all_y_pred, average=None)
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['red' if f < 0.85 else 'steelblue' for f in f1_per_class]
ax.bar(CLASS_NAMES, f1_per_class, color=colors)
ax.axhline(0.85, color='red', linestyle='--', alpha=0.5, label='F1=0.85 target')
ax.set_ylim(0, 1.05)
ax.set_ylabel('F1 score')
ax.set_title('1D-CNN per-class F1 (LOSO)')
ax.legend()
plt.tight_layout()
plt.savefig(DATA_PROC / 'har_f1_per_class.png', dpi=120)
plt.show()